# Data Prep for SVM

In [13]:
import polars as pl

from run_config import PATHS

## Train Test Split

Here we perform the split into train and test data, with a **split** of 70% train, 30% test data. The split is performed **random**. 

In [ ]:
# falls was geändert werden muss schreiben, bitte schreiben 
ALL = True # option all, all possible combination are run
SPATIAL_UNIT = "community" # option: census, community, hexa
TIME_UNIT = "4h" # options: 4h, 2h, 1h
RANDOM = True
OUTPUT = "../data/full/train_test_data/"

In [15]:
if SPATIAL_UNIT not in {"census", "community", "hexa"}:
    SPATIAL_UNIT = "community"
    print("No valid spatial unit was given, default: Community Area was used")

DEMAND_PATHS = {
    "1h": {
        "census": PATHS.gold_1h_demand_census_tracts,
        "community": PATHS.gold_1h_demand_community_areas,
        "hexa": PATHS.gold_1h_demand_hexagon,
    },
    "2h": {
        "census": PATHS.gold_2h_demand_census_tracts,
        "community": PATHS.gold_2h_demand_community_areas,
        "hexa": PATHS.gold_2h_demand_hexagon,
    },
    "4h": {
        "census": PATHS.gold_4h_demand_census_tracts,
        "community": PATHS.gold_4h_demand_community_areas,
        "hexa": PATHS.gold_4h_demand_hexagon,
    },
}


SEED = 42

In [16]:
if ALL:
    time_units = list(DEMAND_PATHS.keys())
    spatial_units = list(next(iter(DEMAND_PATHS.values())).keys())

In [ ]:

if ALL == True:
    for TU in time_units:     
        for SU in spatial_units:
            DATASET = DEMAND_PATHS[TU][SU]
            if(RANDOM == True):
                # split randomly
                df_split = (
                    pl.scan_parquet(DATASET)
                    .with_row_index("_row_id")
                    .with_columns(
                        (pl.col("_row_id").hash(seed=SEED) % 100).alias("_split_bucket")
                    )
                )

                train = (
                    df_split
                    .filter(pl.col("_split_bucket") < 70)
                    .drop(["_row_id", "_split_bucket"])
                )

                test = (
                    df_split
                    .filter(pl.col("_split_bucket") >= 70)
                    .drop(["_row_id", "_split_bucket"])
                )
            else :
                # split according to time
                df_split = pl.scan_parquet(DATASET)

                train = df_split.filter(
                    pl.col("datetime_hour") < pl.datetime(2025, 9, 1)
                )

                test = df_split.filter(
                    pl.col("datetime_hour") >= pl.datetime(2025, 9, 1)
                )


            total_count = df_split.select(pl.len()).collect().item()
            train_count = train.select(pl.len()).collect().item()
            test_count = test.select(pl.len()).collect().item()

            print("Total:", total_count)
            print("Train:", train_count, " Share: ", round(train_count / total_count,2))
            print("Test:", test_count, " Share: ", round(test_count / total_count, 2))

            print("Created parquets for Time Unit:" + TU + "and Spatial Unit:" + SU)

            train.sink_parquet(OUTPUT + "svm_" + SU + "_" + TU + "_train.parquet")
            test.sink_parquet(OUTPUT + "svm_" + SU + "_" + TU + "_test.parquet")
else: 
    if(RANDOM == True):
        # split randomly
        df_split = (
            pl.scan_parquet(DATASET)
            .with_row_index("_row_id")
            .with_columns(
                (pl.col("_row_id").hash(seed=SEED) % 100).alias("_split_bucket")
            )
        )

        train = (
            df_split
            .filter(pl.col("_split_bucket") < 70)
            .drop(["_row_id", "_split_bucket"])
        )

        test = (
            df_split
            .filter(pl.col("_split_bucket") >= 70)
            .drop(["_row_id", "_split_bucket"])
        )
    else :
        # split according to time
        df_split = pl.scan_parquet(DATASET)

        train = df_split.filter(
            pl.col("datetime_hour") < pl.datetime(2025, 9, 1)
        )

        test = df_split.filter(
            pl.col("datetime_hour") >= pl.datetime(2025, 9, 1)
        )


    total_count = df_split.select(pl.len()).collect().item()
    train_count = train.select(pl.len()).collect().item()
    test_count = test.select(pl.len()).collect().item()

    print("Total:", total_count)
    print("Train:", train_count, " Share: ", round(train_count / total_count,2))
    print("Test:", test_count, " Share: ", round(test_count / total_count, 2))

    train.sink_parquet(OUTPUT + "svm_" + SPATIAL_UNIT + "_" + TIME_UNIT + "_train.parquet")
    test.sink_parquet(OUTPUT + "svm_" + SPATIAL_UNIT + "_" + TIME_UNIT + "_test.parquet")

Total: 17932272
Train: 12552667  Share:  0.7
Test: 5379605  Share:  0.3
Total: 1572648
Train: 1099853  Share:  0.7
Test: 472795  Share:  0.3
Total: 17421672
Train: 12195387  Share:  0.7
Test: 5226285  Share:  0.3
Total: 8966136
Train: 6275081  Share:  0.7
Test: 2691055  Share:  0.3
Total: 786324
Train: 550135  Share:  0.7
Test: 236189  Share:  0.3
Total: 8710836
Train: 6096649  Share:  0.7
Test: 2614187  Share:  0.3
Total: 4483068
Train: 3137180  Share:  0.7
Test: 1345888  Share:  0.3
Total: 393162
Train: 274961  Share:  0.7
Test: 118201  Share:  0.3
Total: 4355418
Train: 3047751  Share:  0.7
Test: 1307667  Share:  0.3


In [18]:
df_split.head(10).collect()

_row_id,datetime_hour,month,weekday,hour,month_sin,month_cos,weekday_sin,weekday_cos,hour_sin,hour_cos,tmpc,relh,sknt,vsby,p01m,skyc1_BKN,skyc1_CLR,skyc1_FEW,skyc1_OVC,skyc1_SCT,skyc1_VV,date,is_holiday,h3_cell,food_drink,landmark,shop,train_station,trip_count,trip_seconds_sum,trip_seconds_mean,trip_seconds_min,trip_seconds_max,trip_miles_sum,trip_miles_mean,trip_miles_min,trip_miles_max,fare_sum,fare_mean,fare_min,fare_max,tips_sum,tips_mean,tips_min,tips_max,tolls_sum,tolls_mean,tolls_min,tolls_max,extras_sum,extras_mean,extras_min,extras_max,trip_total_sum,trip_total_mean,trip_total_min,trip_total_max,most_common_payment_type,_split_bucket
u32,datetime[μs],i8,i8,i8,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i8,i8,i8,i8,i8,i8,date,i8,str,f64,f64,f64,f64,u32,i64,f64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,u64
0,2024-04-30 04:00:00,4,2,4,1.0,6.1232e-17,0.781831,0.62349,0.866025,0.5,12.3625,65.765,2.75,10.0,0.0,0,0,1,0,0,0,2024-04-30,0,"""882664cd2dfffff""",3.0,0.0,2.0,0.0,0,0,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""No trips""",55
1,2024-04-30 16:00:00,4,2,16,1.0,6.1232e-17,0.781831,0.62349,-0.866025,-0.5,20.6925,46.765,8.75,10.0,0.0,0,0,1,0,0,0,2024-04-30,0,"""882664cd2dfffff""",3.0,0.0,2.0,0.0,0,0,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""No trips""",30
2,2024-05-19 16:00:00,5,7,16,0.866025,-0.5,-0.781831,0.62349,-0.866025,-0.5,24.0275,35.0825,9.75,10.0,0.0,0,0,1,0,0,0,2024-05-19,0,"""882664cd2dfffff""",3.0,0.0,2.0,0.0,0,0,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""No trips""",67
3,2024-04-27 08:00:00,4,6,8,1.0,6.1232e-17,-0.974928,-0.222521,0.866025,-0.5,17.36,82.9975,11.5,10.0,0.0001,0,0,0,0,1,0,2024-04-27,0,"""882664cd2dfffff""",3.0,0.0,2.0,0.0,0,0,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""No trips""",81
4,2024-05-12 08:00:00,5,7,8,0.866025,-0.5,-0.781831,0.62349,0.866025,-0.5,11.6675,74.91,3.25,10.0,0.0,0,1,0,0,0,0,2024-05-12,0,"""882664cd2dfffff""",3.0,0.0,2.0,0.0,0,0,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""No trips""",82
5,2024-05-25 20:00:00,5,6,20,0.866025,-0.5,-0.974928,-0.222521,-0.866025,0.5,21.1125,32.8925,8.5,10.0,0.0,0,0,1,0,0,0,2024-05-25,0,"""882664cd2dfffff""",3.0,0.0,2.0,0.0,0,0,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""No trips""",13
6,2024-05-14 16:00:00,5,2,16,0.866025,-0.5,0.781831,0.62349,-0.866025,-0.5,16.111667,70.491667,12.666667,10.0,0.0,0,0,1,0,0,0,2024-05-14,0,"""882664cd2dfffff""",3.0,0.0,2.0,0.0,0,0,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""No trips""",50
7,2024-05-28 04:00:00,5,2,4,0.866025,-0.5,0.781831,0.62349,0.866025,0.5,15.4175,77.725,4.75,10.0,0.0,0,0,1,0,0,0,2024-05-28,0,"""882664cd2dfffff""",3.0,0.0,2.0,0.0,0,0,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""No trips""",1
8,2024-05-14 08:00:00,5,2,8,0.866025,-0.5,0.781831,0.62349,0.866025,-0.5,13.055,88.815,8.0,6.25,0.0,0,0,0,1,0,0,2024-05-14,0,"""882664cd2dfffff""",3.0,0.0,2.0,0.0,0,0,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""No trips""",29
